# Earth2Studio Interop Prototype

This notebook demonstrates how seisfetch data feeds into
[NVIDIA Earth2Studio](https://nvidia.github.io/earth2studio/) via the
adapter classes in `seisfetch.earth2`.

Earth2Studio defines protocol-based interfaces:
- **DataSource**: `__call__(time, variable) -> xr.DataArray`
- **DataFrameSource**: `__call__(time, variable, fields) -> pd.DataFrame`

Seismic waveforms are sparse sensor time-series, so both protocols are useful:
- `SeismicDataSource` -> waveform arrays with dims [time, variable, sample]
- `SeismicDataFrameSource` -> per-station summary rows

In [ ]:
from datetime import datetime

from seisfetch import bundle_to_xarray, parse_mseed
from seisfetch.earth2 import (
    SeismicDataFrameSource,
    SeismicDataSource,
    bundle_to_earth2,
)

## 1. Fetch seismic data via S3 (SCEDC)

In [ ]:
from seisfetch import SeisfetchClient

client = SeisfetchClient(provider="SCEDC")
raw = client.get_raw(
    network="CI",
    station="PASC",
    location="00",
    channel="BHZ",
    starttime="2024-01-15T00:00:00",
    endtime="2024-01-15T01:00:00",
)
bundle = parse_mseed(raw)
print(f"Parsed {len(bundle)} traces: {bundle.ids}")

## 2. Seisfetch -> xarray Dataset

In [ ]:
ds = bundle_to_xarray(bundle)
ds

## 3. SeismicDataSource (Earth2Studio DataSource protocol)

Returns `xr.DataArray(dims=[time, variable, sample])`.

In [ ]:
source = SeismicDataSource(bundle)

var_names = list(source._var_names)
print(f"Available variables: {var_names}")

da = source(datetime(2024, 1, 15), var_names)
print(f"Result shape: {da.shape}  dims: {da.dims}")
da

## 4. SeismicDataFrameSource (Earth2Studio DataFrameSource protocol)

Produces a pd.DataFrame with one row per station-channel.

In [ ]:
station_coords = {"CI.PASC": (34.1715, -118.1870)}

df_source = SeismicDataFrameSource(bundle, station_coords=station_coords)
df = df_source(datetime(2024, 1, 15), var_names)
df

## 5. Convenience: bundle_to_earth2

In [ ]:
src = bundle_to_earth2(bundle)
type(src)

## 6. Protocol verification

If earth2studio is installed, verify our adapters satisfy the protocol.

In [ ]:
try:
    from earth2studio.data.base import DataFrameSource, DataSource

    print(f"DataSource:      {isinstance(source, DataSource)}")
    print(f"DataFrameSource: {isinstance(df_source, DataFrameSource)}")
except ImportError:
    print("earth2studio not installed -- skipping protocol check.")
    print("Install with: pip install seisfetch[earth2]")